## Setup
Before running this notebook, install the required dependency:

pip install langchain-text-splitters

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os, json

In [8]:
# Initialize the splitter
# chunk_size is in characters (400 tokens * ~4 chars/token ≈ 1600 chars)
# chunk_overlap is the overlap between adjacent chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1600,
    chunk_overlap=200
)

OUTPUT_CHUNKS_DIR = "data/chunks"
os.makedirs(OUTPUT_CHUNKS_DIR, exist_ok=True)

# ── Text cleaning ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Remove known boilerplate fragments from scraped article text.

    Scraped web pages often contain navigation text, legal disclosures,
    and author bylines that aren't part of the actual article content.
    This function strips those out so the chunks contain only useful
    medical/care information.

    Parameters:
        text: The raw scraped text for one article.

    Returns:
        The cleaned text with boilerplate removed.
    """
    # The ASPCA site includes an affiliate-link disclosure near the top.
    # We find the end of that disclosure and discard everything before it.
    if "aspca.org/chewy" in text:
        disclosure_end_phrase = "no additional cost"
        end_index = text.find(disclosure_end_phrase)
        if end_index != -1:
            # Move past the phrase itself, then strip any leading whitespace.
            text = text[end_index + len(disclosure_end_phrase):].strip()

    # Some articles contain broken "Click here" links left over from HTML
    # that didn't render cleanly. Remove both variants.
    text = text.replace("Click hereto download this medical illustration.", "")
    text = text.replace("Click here", "")

    # Articles end with an author byline section ("WRITTEN BY ...").
    # Everything from that point on is metadata, not content — drop it.
    written_by_marker = "WRITTEN BY"
    if written_by_marker in text:
        text = text[:text.index(written_by_marker)]

    return text.strip()


In [9]:
def chunk_article(article: dict) -> list[dict]:
    """
    Split one article into a list of retrievable text chunks.

    Each chunk is a self-contained piece of the article (roughly 400 tokens)
    with all the article's metadata attached. This metadata lets us show
    the user the title, source, and URL when we retrieve a matching chunk
    later.

    Parameters:
        article: A dictionary representing one scraped article, with keys:
                 url, title, content, source, species, topic.

    Returns:
        A list of chunk dictionaries, one per text segment.
    """
    # Clean the raw text before splitting — garbage in, garbage out.
    cleaned_content = clean_text(article["content"])

    # Split the cleaned text into overlapping chunks.
    text_chunks = splitter.split_text(cleaned_content)

    # Build a unique, human-readable ID for each chunk.
    # Format: "{url}::{species}::chunk_{index}"
    # Including species makes the ID unique even when the same URL is
    # scraped for both dogs and cats (which share many ASPCA pages).
    return [
        {
            "chunk_id": f"{article['url']}::{article['species']}::chunk_{index}",
            "text": chunk,
            "title": article["title"],
            "url": article["url"],
            "source": article["source"],
            "species": article["species"],
            "topic": article["topic"],
        }
        for index, chunk in enumerate(text_chunks)
    ]

In [ ]:
# ── Main processing loop ──────────────────────────────────────────────────────
# Find every JSON file in the raw data folder. Each file is one scraped article.
raw_files = [f for f in os.listdir("data/raw") if f.endswith(".json")]

chunks_path = f"{OUTPUT_CHUNKS_DIR}/all_chunks.json"

# Load any chunks we've already produced from a previous run.
# This lets us add new articles without reprocessing everything.
if os.path.exists(chunks_path):
    with open(chunks_path, encoding="utf-8") as f:
        existing_chunks = json.load(f)
else:
    existing_chunks = []

# Build a set of chunk_ids that already exist.
# A set makes membership checks O(1) — much faster than scanning a list.
existing_chunk_ids = {chunk["chunk_id"] for chunk in existing_chunks}

# Process all articles, generate chunks once per article, keep only new ones.
new_chunks = []
for filename in raw_files:
    with open(os.path.join("data/raw", filename), encoding="utf-8") as f:
        article = json.load(f)

    # Generate chunks once — used for both the duplicate check and storage.
    # The old code called split_text twice per article (once to preview IDs,
    # once inside chunk_article). This does it in a single pass.
    chunks = chunk_article(article)

    truly_new = [c for c in chunks if c["chunk_id"] not in existing_chunk_ids]
    new_chunks.extend(truly_new)

# Combine old and new, then write everything back to disk.
all_chunks = existing_chunks + new_chunks

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

# Summary for the user
print(f"Total raw files:    {len(raw_files)}")
print(f"New chunks added:   {len(new_chunks)}")
print(f"Total chunks:       {len(all_chunks)}")
if raw_files:
    print(f"Avg chunks/article: {len(all_chunks) // len(raw_files)}")

Total raw files:   355
New chunks added:  0
Total chunks:      1597
Avg chunks/article: 4
